# **Model Trigonometri dalam Sistem Navigasi Astronomi untuk Peningkatan Akurasi Posisi Satelit**

In [3]:
import math

## Data

In [4]:
lokasi_pengamatan = ...
local_sidereal_time = ...
suhu_lingkungan = ...
tekanan_udara = ...
azimut = ...
altitude_elevasi = ...
ketinggian_orbit_satelit = ...

## Rumus Rumus

In [ ]:
class Perhitungan():
	def __init__(self, lokasi_pengamatan, local_sidereal_time, suhu_lingkungan, tekanan_udara, azimut, altitude_elevasi, jari_jari_bumi, ketinggian_orbit_satelit):
		self.lokasi_pengamatan = lokasi_pengamatan
		self.local_sidereal_time = local_sidereal_time
		self.suhu_lingkungan = suhu_lingkungan
		self.tekanan_udara = tekanan_udara
		self.azimut = azimut
		self.altitude_elevasi = altitude_elevasi
		self.jari_jari_bumi = jari_jari_bumi
		self.ketinggian_orbit_satelit = ketinggian_orbit_satelit
		self.jari_jari_orbit_satelit = self.jari_jari_bumi + self.ketinggian_orbit_satelit

		self.toposentrik_east_nampak = 1388.550
		self.toposentrik_north_nampak = 1388.550
		self.toposentrik_up_nampak = 526.174

		self.kalkulasi()		



	def kalkulasi(self):
		def sin(degree):
			return math.sin(math.radians(degree))

		def cos(degree):
			return math.cos(math.radians(degree))

		def tan(degree):
			return math.tan(math.radians(degree))

		def solve_abc(a, b, c):
			d = b**2 - 4*a*c
			if d >= 0:
				sol1 = (-b + math.sqrt(d)) / 2*a
				sol2 = (-b - math.sqrt(d)) / 2*a
				return max(sol1, sol2)

		def spatial_error():
			delta_east = self.toposentrik_east_nampak - self.toposentrik_east
			delta_north = self.toposentrik_north_nampak - self.toposentrik_north
			delta_up = self.toposentrik_up_nampak - self.toposentrik_up

			print(delta_east, delta_north, delta_up)

			return math.sqrt(delta_east**2 + delta_north**2 + delta_up**2)

		self.faktor_suhu_tekanan = (self.tekanan_udara / 1010) * (283 / (273 + self.suhu_lingkungan))
		self.faktor_suhu_tekanan = round(self.faktor_suhu_tekanan, 6)
		print(f"Faktor suhu tekanan : {self.faktor_suhu_tekanan}")

		self.argumen_tan = self.altitude_elevasi + (10.3 / (self.altitude_elevasi + 5.11))
		self.argumen_tan = round(self.argumen_tan, 6)
		print(f"Argumen tan : {self.argumen_tan}")

		self.nilai_refraksi = self.faktor_suhu_tekanan * (1.02 / tan(self.argumen_tan)) / 60
		self.nilai_refraksi = round(self.nilai_refraksi, 6)
		print(f"Nilai refraksi : {self.nilai_refraksi}")

		self.a_sebenarnya = self.altitude_elevasi - self.nilai_refraksi
		self.a_sebenarnya = round(self.a_sebenarnya, 6)
		print(f"Nilai a sebanarnya : {self.a_sebenarnya}")

		self.deklinasi = math.degrees(math.asin(sin(self.lokasi_pengamatan) * sin(self.a_sebenarnya) + cos(self.lokasi_pengamatan) * cos(self.a_sebenarnya) * cos(self.azimut)))
		self.deklinasi = round(self.deklinasi,6)
		print(f"Deklinasi : {self.deklinasi}")

		self.hour_angle = math.degrees(math.acos(round((sin(self.a_sebenarnya) - sin(self.lokasi_pengamatan)*sin(self.deklinasi)) / (cos(self.lokasi_pengamatan) * cos(self.deklinasi)), 6)))
		self.hour_angle = round(360 - self.hour_angle, 6) # karena A = 45 derajat, maka H = 360 - H
		print(f"Hour angle : {self.hour_angle}")

		self.asensio_rekta = self.local_sidereal_time - self.hour_angle + 360
		print(f"Asensio rekta : {self.asensio_rekta}")

		self.derivasi_jarak_toposentrik_satelit = solve_abc(1, 2 * self.jari_jari_bumi * math.sin(math.radians(self.a_sebenarnya)), self.jari_jari_bumi**2 - self.jari_jari_orbit_satelit**2)	
		self.derivasi_jarak_toposentrik_satelit = round(self.derivasi_jarak_toposentrik_satelit, 6)
		print(f"Deviasi jarak toposentrik : {self.derivasi_jarak_toposentrik_satelit}")

		self.toposentrik_east = self.derivasi_jarak_toposentrik_satelit * cos(self.a_sebenarnya) * sin(self.azimut)
		self.toposentrik_east = round(self.toposentrik_east, 6)
		print(f"Toposentrik east : {self.toposentrik_east}")

		self.toposentrik_north = self.derivasi_jarak_toposentrik_satelit * cos(self.a_sebenarnya) * cos(self.azimut)
		self.toposentrik_north = round(self.toposentrik_north, 6)
		print(f"Toposentrik north : {self.toposentrik_north}")

		self.toposentrik_up = self.derivasi_jarak_toposentrik_satelit * sin(self.a_sebenarnya)
		self.toposentrik_up = round(self.toposentrik_up, 6)
		print(f"Toposentrik up : {self.toposentrik_up}")

		self.error = spatial_error()
		self.error = round(self.error, 6)
		print(f"Spatial error : {self.error}")
		



In [38]:
Perhitungan(lokasi_pengamatan=-6.2, 
            local_sidereal_time=145, 
            suhu_lingkungan=25, 
            tekanan_udara=1013.25, 
            azimut=45, 
            altitude_elevasi=15,
            jari_jari_bumi=6378.137,
            ketinggian_orbit_satelit=800)

Faktor suhu tekanan : 0.95272
Argumen tan : 15.512183
Nilai refraksi : 0.058354
Nilai a sebanarnya : 14.941646
Deklinasi : 40.643966
Hour angle : 295.789175
Asensio rekta : 209.210825
Deviasi jarak toposentrik : 2036.44653
Toposentrik east : 1391.297704
Toposentrik north : 1391.297704
Toposentrik up : 525.067487
-2.7477040000001125 -2.7477040000001125 1.10651299999995
Spatial error : 4.040313
